# Fibo Lite — capabilities walkthrough

This notebook demonstrates **Fibo Lite** end to end: install the **`fibo-lite`** package from **AWS CodeArtifact** using the Bria Engine, pull model weights from **Hugging Face**, then run the pipeline on three representative flows—**text to image**, **another image from the same VGL document**, and **image plus edit instructions**.

**Prerequisites**

- Linux with a **CUDA** GPU (the stack is aimed at **H100 / H200** class hardware).
- **`BRIA_API_TOKEN`** — used to obtain a short-lived CodeArtifact credential for the **`fibo-lite`** repository.
- **`HF_TOKEN`** — for Hugging Face access to the **`briaai/*`** model repos.


In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads BRIA_API_TOKEN (and friends) from a local .env file, if present

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`fibo-lite`** PyPI repository. The Engine expects your Bria API token in the **`token`** header (set from `BRIA_API_TOKEN`).


In [ ]:
import os

import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
print(f"BRIA_API_TOKEN {BRIA_API_TOKEN}")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

url = "https://engine.prod.bria-api.com/v2/auth/access/code_artifact"
resp = requests.get(
    url,
    params={"repository": "bria-fibo-lite"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
payload = resp.json()

# Engine response shape: { "result": { "authorization_token", "expiration" }, "request_id" }
result = payload.get("result")
auth_token = result.get("authorization_token")
CODE_ARTIFACT_PASSWORD = auth_token.strip()
expiration = result.get("expiration")
print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))

## 2. Install `fibo-lite` from CodeArtifact

Install the package using the **`authorization_token`** from the previous step as the password for the **`bria-fibo-lite`** PyPI simple index (CodeArtifact username **`aws`**).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# Pip imports call os.getcwd(). If Jupyter started from a folder that was later removed, inherited cwd breaks pip.
try:
    _pip_cwd = str(Path.cwd())
except FileNotFoundError:
    _pip_cwd = str(Path.home())
    os.chdir(_pip_cwd)

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
FIBO_LITE_INDEX = "https://aws:" + encoded_password + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-fibo-lite/simple/"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        cwd=_pip_cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install([sys.executable, "-m", "pip", "install", "--upgrade", "requests", "huggingface_hub"])

run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "fibo-lite",
        "--extra-index-url",
        FIBO_LITE_INDEX,
    ]
)

## 3. Download model weights from Hugging Face

Download snapshots for the **VLM** and the **FIBO 1.5** checkpoints using `hf_token`, then pass the **local directories** into the pipeline configuration (`VLMConfig.model_path` and `FiboLiteDiffuserConfig.model_path`).

Repos follow the defaults used in Bria pipelines (`briaai/FIBO-vlm` for the VLM, `briaai/Fibo-1.5` for diffusion).

In [ ]:
import os
from pathlib import Path

from huggingface_hub import snapshot_download

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN for Hugging Face (gated briaai models).")

hf_cache = Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser()


def _download_fibo_lite_weights(token: str, cache_dir: Path) -> tuple[str, str]:
    vlm_path = snapshot_download(repo_id="briaai/FIBO-vlm", token=token, cache_dir=str(cache_dir))
    fibo_path = snapshot_download(repo_id="briaai/Fibo-1.5", token=token, cache_dir=str(cache_dir))
    return vlm_path, fibo_path


vlm_model_path, diffusion_model_path = _download_fibo_lite_weights(hf_token, hf_cache)
print("VLM weights:", vlm_model_path)
print("FIBO 1.5 weights:", diffusion_model_path)

## 4. Build configuration and start the pipeline

`compile_model=True` compiles the diffuser for lower latency after warmup; **`compile_model=False`** skips that path and is often easier for first runs. Tune both flags for your GPU and workload.

TorchInductor cache environment variables below help avoid repeated compilation cost across runs.


In [ ]:
import logging
import os
import time

import torch
from IPython.display import display

from fibo_lite.config import FiboLiteConfig, FiboLiteDiffuserConfig
from fibo_lite.fibo_lite import FiboLite
from fibo_lite.schemas import FiboLiteInput
from fibo_lite.vlm.config import VLMConfig

logging.basicConfig(level=logging.INFO)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Fibo Lite.")

# gpu_memory_utilization is spelled out rather than left to its default: both halves share one GPU,
# and this is the number to turn down first if the diffusion half runs out of memory.
fibo_lite_config = FiboLiteConfig(
    vlm_config=VLMConfig(
        model_path=vlm_model_path,
        max_model_len=1024,
        gpu_memory_utilization=0.4,
    ),
    fibo_lite_config=FiboLiteDiffuserConfig(
        compile_model=False,
        model_path=diffusion_model_path,
    ),
)

pipeline = FiboLite(config=fibo_lite_config)
pipeline.setup()

## 5. Showcase — three Fibo Lite flows

1. **Generate** — generate an image using the FIBO pipeline.
2. **Refine Image Using a VGL Document + Additional Prompt** — use the previously generated VGL document and add a prompt to refine the result with Fibo-Lite.
3. **Generate Image with Input Image from URL** — generate an image using an input image from URL with a prompt to modify it using Fibo-Lite.


In [ ]:
# 5a — Text-to-image (VLM VGL document + generation)

task_generate = FiboLiteInput(
    prompt=("A beautiful sunset over mountains with a lake in the foreground"),
    # aspect_ratio="16:9",  # Options: 1:1, 2:3, 3:2, 3:4, 4:3, 4:5, 5:4, 9:16, 16:9
    # seed= 42  # Optional: for reproducibility
)

t0 = time.time()
out_gen = pipeline.execute(task_generate)
print(f"generate: {time.time() - t0:.1f}s")
display(out_gen.image)

In [ ]:
# 5b - Refine image using a VGL document + additional prompt
refine_prompt = "make the colors more vibrant and add birds in the sky"
task_struct = FiboLiteInput(vgl=out_gen.vgl, prompt=refine_prompt, aspect_ratio="16:9", seed=42)
t0 = time.time()
out_struct = pipeline.execute(task_struct)
print(f"regenerate_from_structured: {time.time() - t0:.1f}s")
display(out_struct.image)

In [ ]:
# 5c — Image + editing instructions
task_img = FiboLiteInput(
    prompt="add a car to the image",
    image="https://bria-test-images.s3.us-east-1.amazonaws.com/highway.jpg",
)
t0 = time.time()
out_img = pipeline.execute(task_img)
print(f"refine_image: {time.time() - t0:.1f}s")
display(out_img.image)

## 6. Cleanup

Release GPU memory held by the pipeline.


In [ ]:
pipeline.cleanup()
print("Cleanup done.")